In [13]:
from pathlib import Path
from itertools import chain, repeat

import pandas as pd
from skimage.measure import regionprops_table
from skimage.io import imread
from nd2 import ND2File

def get_num_positions_nd2(in_file):
    """
    get the number of xy-positions / tiles in an nd2 file, will return 1 if file is just a single (potentially multichannel) stack
    """
    with ND2File(in_file) as reader:
        return reader.sizes['P'] if 'P' in reader.sizes else 1

In [28]:
in_path = '/data/agl_data/NanoFISH/Gabi/GS087_K562-EVI1-GFP_EVI-CTRL/20230927_run0_sd/'

segmentation_subdirectory = 'segmentation_nuclei1_edgesnap'

images_subdirectory = ''

out_subdirectory = 'region_properties'

properties_to_include = ('label', 'area')

In [29]:
mask_files = sorted((Path(in_path) / segmentation_subdirectory).glob('*.tif'))

image_files = sorted((Path(in_path) / images_subdirectory).glob('*.nd2'))

image_files_with_position = list(chain(*(zip(repeat(image_file), range(get_num_positions_nd2(image_file))) for image_file in image_files)))

In [30]:
outdir = Path(in_path) / out_subdirectory

if not outdir.exists():
    outdir.mkdir()

for mask_file, (image_file, position_idx) in zip(mask_files, image_files_with_position):

    mask = imread(mask_file)
    df = pd.DataFrame(regionprops_table(mask, properties=properties_to_include))
    df['image_file'] = image_file
    df['position_idx'] = position_idx

    outfile = outdir / (image_file.stem + '_regionprops.csv')
    df.to_csv(outfile, index=None)